# Berlin multisensor downscaling and temporal mapping

This notebook runs a complete, auditable workflow for Berlin:

- discover and acquire Sentinel-3 LST, Sentinel-2 L2A and Sentinel-5P NO2;
- acquire OpenAQ station observations and create a guarded raster surface;
- harmonize spatial grids and acquisition times;
- train a Random Forest baseline on the observed coarse thermal grid;
- compare OLS, TsHARP/DisTrad and Random Forest on a blocked holdout;
- enforce coarse-scale thermal conservation on the fine prediction;
- inspect temporal layers in an interactive dashboard.

The model output is a prediction, not a new measurement. The notebook keeps
source observations, support masks, residual uncertainty and provenance visible.
Set `SENTINEL_NOTEBOOK_LIVE=1` to execute network acquisition, or leave live
acquisition disabled to reload the latest cubes from `SENTINEL_NOTEBOOK_OUTPUT`.
window spans three days and keeps several acquisitions for the temporal map.
The default uses Sentinel-3 and Sentinel-2 so the temporal map remains
lightweight. Sentinel-5P scenes without valid pixels over the AOI are skipped
without discarding the other sensors; add it with `SENTINEL_NOTEBOOK_SENSORS`.

In [ ]:
from pathlib import Path
import os
import sys
import warnings

import matplotlib.pyplot as plt
import numpy as np
import xarray as xr
from IPython.display import display

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from sentinel_analysis import (
    AnalysisRequest,
    AnalysisResult,
    AnalysisWorkflow,
    AuxiliarySpec,
    ClientConfig,
    fit_coarse_consistent_linear_downscaler,
    fit_coarse_consistent_random_forest_downscaler,
    fit_coarse_consistent_tsharp_downscaler,
    fit_linear_downscaler,
    fit_random_forest_downscaler,
    fit_tsharp_downscaler,
    split_spatiotemporal,
    validate_downscaler,
    get_city,
    OpenAQProvider,
)

RUN_LIVE = os.getenv('SENTINEL_NOTEBOOK_LIVE', '0') == '1'
LOAD_EXISTING = os.getenv('SENTINEL_NOTEBOOK_LOAD_EXISTING', '1') == '1'
SENSORS = tuple(value.strip().lower() for value in os.getenv('SENTINEL_NOTEBOOK_SENSORS', 'sentinel3,sentinel2').split(',') if value.strip())
START_DATE = os.getenv('SENTINEL_NOTEBOOK_START', '2025-06-16')
END_DATE = os.getenv('SENTINEL_NOTEBOOK_END', '2025-06-18')
MAX_PRODUCTS = int(os.getenv('SENTINEL_NOTEBOOK_MAX_PRODUCTS', '8'))
PREDICTOR_RESOLUTION_M = int(os.getenv('SENTINEL_NOTEBOOK_RESOLUTION_M', '100'))
OUTPUT_DIR = Path(os.getenv('SENTINEL_NOTEBOOK_OUTPUT', str(PROJECT_ROOT / 'output' / 'berlin-multisensor-notebook')))
if not OUTPUT_DIR.is_absolute():
    OUTPUT_DIR = PROJECT_ROOT / OUTPUT_DIR
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CITY = get_city('Berlin')
print({
    'live': RUN_LIVE,
    'sensors': SENSORS,
    'period': (START_DATE, END_DATE),
    'max_products_per_sensor': MAX_PRODUCTS,
    'predictor_resolution_m': PREDICTOR_RESOLUTION_M,
    'output': str(OUTPUT_DIR),
})

## 1. Request and acquisition

The request uses Sentinel-3 as the thermal target and a finer predictor grid.
OpenAQ interpolation is deliberately configured with a minimum support of three
stations and a 10 km radius. Unsupported cells remain missing.

In [ ]:
request = AnalysisRequest.for_city(
    CITY,
    START_DATE,
    END_DATE,
    sensors=SENSORS,
    variables=('lst', 'NO2'),
    resolution_m=PREDICTOR_RESOLUTION_M,
    max_products_per_sensor=MAX_PRODUCTS,
    auxiliary=(
        AuxiliarySpec(
            'openaq',
            variables=('NO2',),
            options={
                'max_locations': 100,
                'interpolation_method': 'idw',
                'interpolation_power': 2.0,
                'interpolation_max_distance_m': 10000,
                'interpolation_min_neighbors': 3,
                'interpolation_max_neighbors': 12,
            },
        ),
    ),
)

if RUN_LIVE:
    result = AnalysisWorkflow(request).execute(
        OUTPUT_DIR,
        config=ClientConfig.from_env(),
        max_workers=1,
    )
    print('Acquisition and fusion completed')
elif LOAD_EXISTING and (OUTPUT_DIR / 'thermal_fused_cube.nc').exists() and (OUTPUT_DIR / 'predictor_cube.nc').exists():
    cube = xr.load_dataset(OUTPUT_DIR / 'thermal_fused_cube.nc')
    predictor_cube = xr.load_dataset(OUTPUT_DIR / 'predictor_cube.nc')
    station_files = sorted((OUTPUT_DIR / 'auxiliary' / 'openaq').glob('*.json'))
    auxiliary = {'openaq': OpenAQProvider().open(station_files[0])} if station_files else None
    result = AnalysisResult(cube=cube, plan=AnalysisWorkflow(request).plan, provenance={'loaded_from': str(OUTPUT_DIR)}, thermal_cube=cube, predictor_cube=predictor_cube, auxiliary=auxiliary)
    print(f'Loaded existing analysis from {OUTPUT_DIR}')
else:
    result = None
    print('Live acquisition disabled and no reusable output was found.')

In [ ]:
if result is not None:
    cube = result.cube
    predictor_cube = result.predictor_cube
    print('thermal cube:', dict(cube.sizes))
    print('cube variables:', sorted(cube.data_vars))
    print('predictor cube:', None if predictor_cube is None else dict(predictor_cube.sizes))
    print('provenance:', result.provenance)
    if result.auxiliary and 'openaq' in result.auxiliary:
        print('OpenAQ station table:', dict(result.auxiliary['openaq'].sizes))
else:
    print('Run the acquisition cell first with SENTINEL_NOTEBOOK_LIVE=1.')

## 2. Coarse-consistent downscaling

The training table is the observed Sentinel-3 grid with collocated predictors.
The model is then applied to the finer predictor cube. The correction step forces
the fine result to reaggregate to the observed coarse LST. This notebook is a
demonstration, not a validated production model. Use more dates,
blocked spatial holdouts and independent thermal references for scientific use.

In [ ]:
if result is not None and predictor_cube is not None:
    # The fused cube already contains predictors collocated to the observed thermal grid.
    thermal_training = cube
    training_cube = cube
    preferred = ('NDVI', 'NDBI', 'EVI', 'NDMI', 'NO2')
    predictor_names = [
        name for name in preferred
        if name in training_cube and name in predictor_cube
        and bool(np.isfinite(training_cube[name].values).any())
        and bool(np.isfinite(predictor_cube[name].values).any())
    ]
    if len(predictor_names) < 2:
        raise RuntimeError(f'Not enough collocated predictors: {predictor_names}')
    print('predictors:', predictor_names)
    try:
        model = fit_coarse_consistent_random_forest_downscaler(
            training_cube,
            predictors=tuple(predictor_names),
            n_estimators=200,
            random_state=42,
            min_samples=100,
            conservation_tolerance=0.25,
        )
        print('model: coarse-consistent Random Forest')
    except RuntimeError as exc:
        warnings.warn(f'Random Forest unavailable ({exc}); using OLS baseline')
        model = fit_coarse_consistent_linear_downscaler(
            training_cube, predictors=tuple(predictor_names), min_samples=100, conservation_tolerance=0.25
        )
        print('model: coarse-consistent OLS')
    # Keep the fine predictor dates; the display layer aligns them to thermal dates below.
    downscaled = model.predict(predictor_cube, thermal_training)
    display_downscaled = downscaled.reindex(
        time=cube.time, method='nearest', tolerance=str(request.temporal_tolerance)
    )
    display_predictors = predictor_cube.reindex(
        time=cube.time, method='nearest', tolerance=str(request.temporal_tolerance)
    )
    print({key: value for key, value in downscaled.attrs.items() if key.startswith('coarse_')})
    # Benchmark candidate relationships on a future-time/spatial holdout.
    blocked_metrics = {}
    try:
        train_block, validation_block = split_spatiotemporal(training_cube, validation_fraction=0.2, block_period=5)
        candidates = {
            'OLS': lambda: fit_linear_downscaler(train_block, predictors=tuple(predictor_names), min_samples=100),
            'TsHARP/DisTrad': lambda: fit_tsharp_downscaler(train_block, predictor=predictor_names[0], min_samples=100),
        }
        if hasattr(model.base_model, 'random_state'):
            candidates['Random Forest'] = lambda: fit_random_forest_downscaler(train_block, predictors=tuple(predictor_names), n_estimators=200, random_state=42, min_samples=100)
        for label, factory in candidates.items():
            try:
                blocked_metrics[label] = validate_downscaler(factory(), validation_block)
            except (RuntimeError, ValueError) as exc:
                warnings.warn(f'{label} benchmark unavailable: {exc}')
        print('blocked benchmark metrics:', blocked_metrics)
    except (RuntimeError, ValueError) as exc:
        warnings.warn(f'Blocked benchmark unavailable for this scene set: {exc}')
else:
    downscaled = None
    display_downscaled = None
    display_predictors = None
    blocked_metrics = {}
    print('Downscaling waits for a live result.')

## 3. Berlin map layers

The dashboard uses projected coordinates from the analysis grid. It shows the
observed thermal field, the modelled fine thermal field, optical vegetation and
built-up indices, Sentinel-5P NO2, the guarded OpenAQ surface and interpolation
support. The time slider changes the thermal and satellite acquisition frame.

In [ ]:
def _celsius(data):
    units = str(data.attrs.get('units', '')).lower()
    return data - 273.15 if units in {'k', 'kelvin'} or (units == '' and float(data.median()) > 200) else data

def _frame(data, index):
    if data is None:
        return None
    return data.isel(time=min(index, data.sizes.get('time', 1) - 1)) if 'time' in data.dims else data

def _plot_layer(ax, data, index, title, cmap, units=None, vmin=None, vmax=None):
    if data is None:
        ax.set_title(f'{title}\nnot available')
        ax.axis('off')
        return
    layer = _frame(data, index)
    mesh = ax.pcolormesh(layer.x, layer.y, layer, shading='auto', cmap=cmap, vmin=vmin, vmax=vmax)
    ax.set_title(title)
    ax.set_aspect('equal')
    ax.set_xlabel('Easting (m)')
    ax.set_ylabel('Northing (m)')
    plt.colorbar(mesh, ax=ax, shrink=0.78, label=units or str(data.attrs.get('units', '')))

def plot_berlin_dashboard(index=0):
    if result is None or downscaled is None:
        print('Run the live acquisition and downscaling cells first.')
        return
    fig, axes = plt.subplots(2, 4, figsize=(20, 10), constrained_layout=True)
    lst = _celsius(cube['lst']) if 'lst' in cube else None
    predicted = _celsius(display_downscaled['lst_downscaled'])
    ndvi = display_predictors['NDVI'] if 'NDVI' in display_predictors else None
    ndbi = display_predictors['NDBI'] if 'NDBI' in display_predictors else None
    s5p = cube['NO2'] if 'NO2' in cube else None
    openaq = cube['auxiliary_openaq_NO2'] if 'auxiliary_openaq_NO2' in cube else None
    support = cube['auxiliary_openaq_NO2_interpolation_valid'] if 'auxiliary_openaq_NO2_interpolation_valid' in cube else None
    _plot_layer(axes[0, 0], lst, index, 'Sentinel-3 observed LST', 'inferno', 'deg C')
    _plot_layer(axes[0, 1], predicted, index, 'Coarse-consistent downscaled LST', 'inferno', 'deg C')
    _plot_layer(axes[0, 2], ndvi, index, 'Sentinel-2 NDVI', 'YlGn', '1', -1, 1)
    _plot_layer(axes[0, 3], ndbi, index, 'Sentinel-2 NDBI', 'magma', '1', -1, 1)
    _plot_layer(axes[1, 0], s5p, index, 'Sentinel-5P NO2 column', 'viridis', 'mol m-2')
    _plot_layer(axes[1, 1], openaq, index, 'OpenAQ guarded IDW surface', 'plasma', 'ug m-3')
    _plot_layer(axes[1, 2], support, index, 'OpenAQ interpolation support', 'Blues', 'valid')
    axes[1, 3].axis('off')
    axes[1, 3].text(0.02, 0.95, f'Frame {index}', va='top', fontsize=14)
    axes[1, 3].text(0.02, 0.82, 'Observed layers are not modelled.\nThe downscaled LST is modelled\nand coarse-scale constrained.', va='top')
    return fig

if result is not None:
    display(plot_berlin_dashboard(0))

In [ ]:
if result is not None and os.getenv('SENTINEL_NOTEBOOK_INTERACTIVE', '0') == '1':
    try:
        from ipywidgets import IntSlider, interact
        frame_count = int(cube.sizes.get('time', 1))
        interact(plot_berlin_dashboard, index=IntSlider(min=0, max=max(0, frame_count - 1), step=1, value=0, description='time'))
    except ImportError:
        print('Install ipywidgets for the temporal slider; static frame 0 was rendered above.')

## 4. Save analysis products

The saved files contain the observed cube, predictor cube, modelled output and
the request/provenance metadata. The source archives and manifests remain in the
same output directory and should be retained for reproducibility.

In [ ]:
def _netcdf_safe(dataset):
    def safe_attrs(attrs):
        return {
            key: int(value) if isinstance(value, (bool, np.bool_)) else str(value) if isinstance(value, (dict, list, tuple)) else value
            for key, value in attrs.items()
        }
    clean = dataset.copy()
    clean.attrs = safe_attrs(clean.attrs)
    for name in clean.variables:
        clean[name].attrs = safe_attrs(clean[name].attrs)
    return clean

if result is not None and downscaled is not None:
    _netcdf_safe(cube).to_netcdf(OUTPUT_DIR / 'thermal_fused_cube.nc')
    _netcdf_safe(predictor_cube).to_netcdf(OUTPUT_DIR / 'predictor_cube.nc')
    _netcdf_safe(downscaled).to_netcdf(OUTPUT_DIR / 'coarse_consistent_downscaled_lst.nc')
    figure = plot_berlin_dashboard(0)
    figure.savefig(OUTPUT_DIR / 'berlin_multisensor_dashboard.png', dpi=180)
    plt.close(figure)
    from PIL import Image
    import io
    frames = []
    for frame_index in range(int(cube.sizes.get('time', 1))):
        frame, axes = plt.subplots(1, 2, figsize=(10, 5), constrained_layout=True)
        for axis, data, title in ((axes[0], _celsius(cube['lst']), 'Sentinel-3 observed LST'), (axes[1], _celsius(display_downscaled['lst_downscaled']), 'Downscaled LST')):
            layer = _frame(data, frame_index)
            mesh = axis.pcolormesh(layer.x, layer.y, layer, shading='auto', cmap='inferno')
            axis.set_title(title)
            axis.set_aspect('equal')
            frame.colorbar(mesh, ax=axis, shrink=0.8, label='deg C')
        buffer = io.BytesIO()
        frame.savefig(buffer, format='png', dpi=75)
        plt.close(frame)
        buffer.seek(0)
        frames.append(Image.open(buffer).convert('P').copy())
    animation_path = OUTPUT_DIR / 'berlin_multisensor_animation.gif'
    if frames:
        frames[0].save(animation_path, save_all=True, append_images=frames[1:], duration=900, loop=0, optimize=False)
        from IPython.display import Image as NotebookImage
        display(NotebookImage(filename=str(animation_path), embed=True))
    print('saved:', sorted(path.name for path in OUTPUT_DIR.glob('*.nc')) + ['berlin_multisensor_dashboard.png', 'berlin_multisensor_animation.gif'])
else:
    print('Nothing saved because live acquisition was disabled.')